# GeoPlan Multi-Agent Workflow

This notebook connects the deterministic GeoPlan tools to a
LlamaIndex workflow using a local Ollama model.

The GIS pipeline remains the source of truth for candidate IDs,
scores, distances, selected-site order, and constraint checks.
Agents retrieve evidence, compare candidates, review limitations,
and produce a structured planning report.

Run Jupyter from the repository root and complete notebooks 01 and
02 before running this notebook.


## 2. Imports and project setup


In [2]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import math
import re
import sys
import traceback

import pandas as pd
from IPython.display import display
from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    ValidationError,
)

from llama_index.llms.ollama import Ollama
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    ReActAgent,
    AgentInput,
    AgentOutput,
    AgentStream,
    ToolCall,
    ToolCallResult,
)


In [3]:
PROJECT_ROOT = Path.cwd().resolve()
SOURCE_DIR = PROJECT_ROOT / "src"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from geoplan_agent_tools import GeoPlanToolbox

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


## 3. Load the deterministic GeoPlan tools


In [4]:
toolbox = GeoPlanToolbox(
    PROJECT_ROOT,
    strict=True,
)

tools = toolbox.create_llamaindex_tools()


## 4. Configure the local Ollama model

Start Ollama in a terminal:

```bash
ollama serve
ollama list
```

The model name must exactly match a tag shown by `ollama list`.


In [5]:
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b-instruct"
REQUEST_TIMEOUT_SECONDS = 600.0
WORKFLOW_TIMEOUT_SECONDS = 1800.0


In [6]:
llm = Ollama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    request_timeout=REQUEST_TIMEOUT_SECONDS,
    context_window=8096,
    temperature=0.0,
    keep_alive="0m",
)


## 6. Pydantic output schemas

The final workflow output must validate against `FinalPlanningReport`.


In [7]:
class CandidateRecommendation(BaseModel):
    model_config = ConfigDict(extra="forbid")

    candidate_id: str = Field(
        description="Exact candidate ID returned by a GeoPlan tool."
    )
    address: str | None = Field(
        default=None,
        description="Exact address returned by a GeoPlan tool.",
    )
    selection_round: int | None = Field(
        default=None,
        description=(
            "Exact sequential selection round from "
            "selected_site_sequence.csv."
        ),
    )
    selection_score: float | None = Field(
        default=None,
        description=(
            "Exact selection score from selected_site_sequence.csv."
        ),
    )
    marginal_population: float | None = Field(
        default=None,
        description=(
            "Exact marginal population from "
            "selected_site_sequence.csv."
        ),
    )
    capacity_filled: float | None = Field(
        default=None,
        description=(
            "Exact capacity value returned by the deterministic tools."
        ),
    )
    overall_score: float | None = Field(
        default=None,
        description=(
            "Exact overall score returned by the deterministic tools."
        ),
    )
    strengths: list[str] = Field(
        default_factory=list,
        description=(
            "Only evidence-supported statements. Include the metric "
            "name and exact value in every strength."
        ),
    )
    concerns: list[str] = Field(
        default_factory=list,
        description=(
            "Only tool-supported warnings or limitations. "
            "Do not infer engineering problems."
        ),
    )
    evidence_sources: list[str] = Field(
        default_factory=list,
        description=(
            "Tool or exported-file names supporting this record."
        ),
    )


class SpecialistFinding(BaseModel):
    model_config = ConfigDict(extra="forbid")

    agent_name: str
    findings: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)
    evidence_tools: list[str] = Field(default_factory=list)


class FinalPlanningReport(BaseModel):
    model_config = ConfigDict(extra="forbid")

    objective: str
    scenario: str
    selection_method: str
    requested_site_count: int
    recommended_sites: list[CandidateRecommendation]
    specialist_findings: list[SpecialistFinding] = Field(
        default_factory=list
    )
    robustness_findings: list[str] = Field(
        default_factory=list
    )
    data_quality_warnings: list[str] = Field(
        default_factory=list
    )
    limitations: list[str] = Field(
        default_factory=list
    )
    reviewer_approved: bool


## 7. Create the specialist agents


In [8]:
site_evidence_agent = ReActAgent(
    name="SiteEvidenceAgent",
    description=(
        "Retrieves and explains deterministic candidate demand, "
        "coverage, accessibility, and feasibility evidence."
    ),
    system_prompt=(
        "You are the SiteEvidenceAgent. Your scope is candidate "
        "evidence only. "

        "MANDATORY PROCESS: "
        "1. Call get_selected_sequence with the requested site count. "
        "2. Use get_candidate for candidate-summary evidence when needed. "
        "3. Use compare_candidates only with fields listed by "
        "list_available_metrics. "

        "IMPORTANT FIELD RULE: marginal_population and selection_score "
        "come from get_selected_sequence. They are not valid "
        "compare_candidates metrics. population_1000m is a different "
        "metric and must not be presented as marginal_population. "

        "GROUNDING RULES: "
        "- Preserve candidate IDs, addresses, rounds, and numbers exactly. "
        "- Every strength must name a metric and its exact tool-returned "
        "value. "
        "- Do not write vague claims such as 'high-demand urban area', "
        "'publicly available lot', 'good accessibility', or "
        "'close to traffic' unless a named metric and value support it. "
        "- Do not convert screening scores into engineering conclusions. "
        "- Do not invent unavailable fields. "
        "- Do not perform scenario analysis or final review. "

        "HANDOFF RULE: your final action must be handoff to "
        "PlanningCoordinator. In the handoff reason, provide a compact "
        "evidence table for every selected ID containing: candidate_id, "
        "address, selection_round, selection_score, marginal_population, "
        "capacity_filled, overall_score, exact strengths, exact concerns, "
        "and tools used. Do not answer the user directly."
    ),
    tools=[
        tools["list_available_metrics"],
        tools["list_candidate_ids"],
        tools["get_candidate"],
        tools["get_top_candidates"],
        tools["compare_candidates"],
        tools["get_selected_sequence"],
    ],
    can_handoff_to=["PlanningCoordinator"],
    llm=llm,
    streaming=True,
)


In [9]:
scenario_risk_agent = ReActAgent(
    name="ScenarioRiskAgent",
    description=(
        "Reviews scenario robustness, sensitivity, candidate audits, "
        "and data-quality limitations."
    ),
    system_prompt=(
        "You are the ScenarioRiskAgent. Your scope is robustness, "
        "risk, and data quality only. "

        "MANDATORY PROCESS: "
        "1. Call get_scenario_results for the requested scenario. "
        "2. Call get_sensitivity_results. "
        "3. Call audit_recommendation_set for the exact selected IDs. "
        "4. Call audit_candidate only when candidate-level details are "
        "needed. "

        "GROUNDING RULES: "
        "- When a scenario or sensitivity tool returns unavailable, say "
        "exactly that the analysis was not available. Do not infer "
        "robustness. "
        "- Do not say a field is unavailable before checking the tool. "
        "- existing_chargers_2000m and traffic_data_quality may be in "
        "get_candidate; retrieve them before discussing them. "
        "- Preserve all deterministic warnings and critical issues. "
        "- Screening feasibility is not electrical, construction, "
        "ownership, utility, or permitting feasibility. "
        "- Do not introduce new candidate IDs or change the selection. "
        "- Do not answer the user directly. "

        "HANDOFF RULE: your final action must be handoff to "
        "PlanningCoordinator. Include: scenario-tool status, "
        "sensitivity-tool status, exact audit decision, critical issues, "
        "warnings, spacing result, and tools used."
    ),
    tools=[
        tools["get_candidate"],
        tools["compare_candidates"],
        tools["get_scenario_results"],
        tools["get_sensitivity_results"],
        tools["audit_candidate"],
        tools["audit_recommendation_set"],
    ],
    can_handoff_to=["PlanningCoordinator"],
    llm=llm,
    streaming=True,
)


In [10]:
final_reviewer_agent = ReActAgent(
    name="FinalReviewerAgent",
    description=(
        "Performs deterministic and semantic quality control on the "
        "proposed planning report."
    ),
    system_prompt=(
        "You are the FinalReviewerAgent. Your scope is final quality "
        "control only. "

        "MANDATORY PROCESS: "
        "1. Call get_planning_configuration. "
        "2. Call get_selected_sequence for the requested count. "
        "3. Call audit_recommendation_set for the exact proposed IDs. "
        "4. Use get_candidate to verify any disputed candidate fact. "

        "REVIEW CHECKLIST: "
        "- IDs exactly match the deterministic selected sequence. "
        "- Count, uniqueness, addresses, and selection rounds agree. "
        "- The scenario equals the user-requested scenario. "
        "- 'deterministic selected sequence' is the selection method, "
        "not the scenario name. "
        "- marginal_population is described as available from the "
        "selected sequence, not unavailable globally. "
        "- Every numeric claim is present in a tool result. "
        "- Vague unsupported claims are removed. "
        "- Limitations are non-empty. "
        "- Engineering feasibility is not claimed. "
        "- Reviewer approval agrees with audit_recommendation_set. "

        "DECISION RULE: APPROVED only when the deterministic audit has "
        "reviewer_approved=true, no critical issues remain, and the "
        "report is internally consistent. Otherwise return "
        "REVISION_REQUIRED. "

        "HANDOFF RULE: your final action must be handoff to "
        "PlanningCoordinator. Include the decision, exact critical "
        "issues, warnings, spacing result, factual corrections, and tools "
        "used. Do not write the user-facing final report."
    ),
    tools=[
        tools["get_planning_configuration"],
        tools["get_candidate"],
        tools["get_selected_sequence"],
        tools["audit_candidate"],
        tools["audit_recommendation_set"],
    ],
    can_handoff_to=["PlanningCoordinator"],
    llm=llm,
    streaming=True,
)


## 8. Create the coordinator

The coordinator must delegate in this order:

1. `SiteEvidenceAgent`
2. `ScenarioRiskAgent`
3. `FinalReviewerAgent`
4. final structured answer


In [11]:
planning_coordinator = ReActAgent(
    name="PlanningCoordinator",
    description=(
        "Coordinates GeoPlan specialists and writes the final "
        "evidence-grounded report."
    ),
    system_prompt=(
        "You are the root PlanningCoordinator. You coordinate agents and "
        "produce the final FinalPlanningReport. "

        "MANDATORY ORDER: "
        "1. Hand off to SiteEvidenceAgent. "
        "2. After control returns, hand off to ScenarioRiskAgent. "
        "3. After control returns, hand off to FinalReviewerAgent. "
        "4. After control returns, produce the final structured report. "
        "Never skip a specialist and never finish immediately after a "
        "handoff. "

        "FINAL REPORT CONSISTENCY RULES: "
        "- scenario must equal the scenario explicitly requested by the "
        "user, for example 'balanced'. "
        "- selection_method must be 'deterministic selected sequence'. "
        "- requested_site_count must equal the user's requested count. "
        "- recommended_sites must preserve the exact selected sequence "
        "order and exact tool-returned IDs. "
        "- Copy address, selection_round, selection_score, "
        "marginal_population, capacity_filled, and overall_score exactly "
        "from specialist/tool evidence. "
        "- Every strength must name a metric and exact value. "
        "- Every concern must come from an audit, missing optional "
        "analysis, or a stated model limitation. "
        "- Do not call marginal_population unavailable: it is available "
        "from get_selected_sequence, although it is not a valid "
        "compare_candidates field. "
        "- robustness_findings must be empty or explicitly state "
        "'not evaluated because the export was unavailable' when the "
        "scenario or sensitivity tool was unavailable. "
        "- limitations must contain at least three concrete items: "
        "engineering feasibility not assessed; utility/electrical "
        "capacity not assessed; construction cost/ownership/permitting "
        "not assessed. "
        "- reviewer_approved must match the FinalReviewerAgent decision "
        "and deterministic audit. "
        "- Do not add general geographic or infrastructure claims that "
        "were not returned by a tool. "

        "Use compact wording because the local model has limited context. "
        "Return only the FinalPlanningReport-compatible final response."
    ),
    tools=[],
    can_handoff_to=[
        "SiteEvidenceAgent",
        "ScenarioRiskAgent",
        "FinalReviewerAgent",
    ],
    llm=llm,
    streaming=True,
)


## 9. Assemble `AgentWorkflow`


In [12]:
agent_workflow = AgentWorkflow(
    agents=[
        planning_coordinator,
        site_evidence_agent,
        scenario_risk_agent,
        final_reviewer_agent,
    ],
    root_agent=planning_coordinator.name,
    initial_state={
        "workflow_name": "GeoPlan Multi-Agent Review",
        "evidence_complete": False,
        "risk_review_complete": False,
        "final_review_complete": False,
        "revision_count": 0,
    },
    output_cls=FinalPlanningReport,
    timeout=WORKFLOW_TIMEOUT_SECONDS,
    early_stopping_method="generate",
)

print("Root agent:", planning_coordinator.name)


Root agent: PlanningCoordinator


## 10. Stream and save the event trace


In [13]:
def preview(value: Any, limit: int = 1500) -> str:
    try:
        text = json.dumps(value, default=str, ensure_ascii=False)
    except TypeError:
        text = str(value)
    return text[:limit]


async def run_with_trace(
    workflow: AgentWorkflow,
    user_message: str,
    max_iterations: int = 40,
):
    handler = workflow.run(
        user_msg=user_message,
        max_iterations=max_iterations,
        early_stopping_method="generate",
    )

    rows = []
    current_agent = None
    step = 0

    async for event in handler.stream_events():
        step += 1
        event_agent = getattr(
            event,
            "current_agent_name",
            current_agent,
        )

        if event_agent and event_agent != current_agent:
            current_agent = event_agent
            print("\n" + "=" * 60)
            print("Agent:", current_agent)
            print("=" * 60)
            rows.append({
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "step": step,
                "agent": current_agent,
                "event_type": "AgentTransition",
                "tool_name": None,
                "arguments": None,
                "output_preview": None,
            })

        if isinstance(event, AgentStream):
            if event.delta:
                print(event.delta, end="", flush=True)

        elif isinstance(event, ToolCall):
            print("\nCalling tool:", event.tool_name)
            print("Arguments:", event.tool_kwargs)
            rows.append({
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "step": step,
                "agent": current_agent,
                "event_type": "ToolCall",
                "tool_name": event.tool_name,
                "arguments": preview(event.tool_kwargs),
                "output_preview": None,
            })

        elif isinstance(event, ToolCallResult):
            print("\nTool result:", event.tool_name)
            print(str(event.tool_output)[:1000])
            rows.append({
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "step": step,
                "agent": current_agent,
                "event_type": "ToolCallResult",
                "tool_name": event.tool_name,
                "arguments": preview(event.tool_kwargs),
                "output_preview": str(event.tool_output)[:1500],
            })

        elif isinstance(event, AgentOutput):
            content = event.response.content if event.response else None
            planned = [
                call.tool_name for call in (event.tool_calls or [])
            ]
            if planned:
                print("\nPlanned tools:", planned)
            rows.append({
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "step": step,
                "agent": current_agent,
                "event_type": "AgentOutput",
                "tool_name": ", ".join(planned) if planned else None,
                "arguments": None,
                "output_preview": str(content)[:1500] if content else None,
            })

    response = await handler
    return response, pd.DataFrame(rows)


## 11. Run one complete recommendation request

The first demo uses five sites so the 4B model receives a manageable amount of
tool output. Increase the count only after the full workflow succeeds.


In [14]:
DEMO_SITE_COUNT = 5
DEMO_SCENARIO = "balanced"

DEMO_REQUEST = f"""
Recommend exactly {DEMO_SITE_COUNT} Toronto public EV-charging
candidate sites.

Required scenario:
{DEMO_SCENARIO}

Required selection method:
deterministic selected sequence

Required workflow:
1. SiteEvidenceAgent retrieves exact selected-site evidence.
2. ScenarioRiskAgent checks scenario, sensitivity, audit, and risk status.
3. FinalReviewerAgent validates the exact recommendation set.
4. PlanningCoordinator returns FinalPlanningReport.

Evidence rules:
- Every candidate ID, address, round, and number must come from a tool.
- Keep scenario='{DEMO_SCENARIO}'.
- Do not call the deterministic selected sequence a scenario.
- marginal_population comes from get_selected_sequence.
- Every strength must state a metric name and exact value.
- Unsupported descriptive claims are prohibited.
- Include at least three concrete limitations.
- This is planning screening, not confirmed engineering feasibility.
"""

print(DEMO_REQUEST)



Recommend exactly 5 Toronto public EV-charging
candidate sites.

Required scenario:
balanced

Required selection method:
deterministic selected sequence

Required workflow:
1. SiteEvidenceAgent retrieves exact selected-site evidence.
2. ScenarioRiskAgent checks scenario, sensitivity, audit, and risk status.
3. FinalReviewerAgent validates the exact recommendation set.
4. PlanningCoordinator returns FinalPlanningReport.

Evidence rules:
- Every candidate ID, address, round, and number must come from a tool.
- Keep scenario='balanced'.
- Do not call the deterministic selected sequence a scenario.
- marginal_population comes from get_selected_sequence.
- Every strength must state a metric name and exact value.
- Unsupported descriptive claims are prohibited.
- Include at least three concrete limitations.
- This is planning screening, not confirmed engineering feasibility.



In [15]:
try:
    workflow_response, multiagent_trace = await run_with_trace(
        agent_workflow,
        DEMO_REQUEST,
        max_iterations=40,
    )
except Exception:
    traceback.print_exc()
    raise

TRACE_PATH = OUTPUT_DIR / "multiagent_trace.csv"
multiagent_trace.to_csv(TRACE_PATH, index=False)

print("\nSaved trace:", TRACE_PATH.resolve())
display(multiagent_trace)



Agent: PlanningCoordinator
```
Thought: The current language of the user is: English. I need to use a tool to help me answer the question. The first step in the workflow is to hand off to SiteEvidenceAgent to retrieve exact selected-site evidence for 5 Toronto public EV-charging candidate sites under the balanced scenario.
Action: handoff
Action Input: {"to_agent": "SiteEvidenceAgent", "reason": "To retrieve exact selected-site evidence for 5 Toronto public EV-charging candidate sites under the balanced scenario as per the required workflow."}
```
Planned tools: ['handoff']

Calling tool: handoff
Arguments: {'to_agent': 'SiteEvidenceAgent', 'reason': 'To retrieve exact selected-site evidence for 5 Toronto public EV-charging candidate sites under the balanced scenario as per the required workflow.'}

Tool result: handoff
Agent SiteEvidenceAgent is now handling the request due to the following reason: To retrieve exact selected-site evidence for 5 Toronto public EV-charging candidate si

/Users/miladsaeedi/miniforge3/envs/huggingface/lib/python3.10/site-packages/workflows/runtime/types/step_function.py:233: UserWarning: There was a problem with the generation of the structured output: {"error":{"code":400,"message":"request (9117 tokens) exceeds the available context size (8192 tokens), try increasing it","type":"exceed_context_size_error","n_prompt_tokens":9117,"n_ctx":8192}} (status code: 400)
  step_result = await call_func(*args, **kwargs)


,timestamp_utc,step,agent,event_type,tool_name,arguments,output_preview
0,2026-07-28T01:32:29.417076+00:00,1,PlanningCoordinator,AgentTransition,None,None,None
1,2026-07-28T01:32:50.118036+00:00,114,PlanningCoordinator,AgentOutput,handoff,None,```\nThought: The current language of the user...
2,2026-07-28T01:32:50.430860+00:00,115,PlanningCoordinator,ToolCall,handoff,"{""to_agent"": ""SiteEvidenceAgent"", ""reason"": ""T...",None
3,2026-07-28T01:32:50.431169+00:00,116,PlanningCoordinator,ToolCallResult,handoff,"{""to_agent"": ""SiteEvidenceAgent"", ""reason"": ""T...",Agent SiteEvidenceAgent is now handling the re...
4,2026-07-28T01:32:50.819231+00:00,117,SiteEvidenceAgent,AgentTransition,None,None,None
5,2026-07-28T01:33:10.029269+00:00,202,SiteEvidenceAgent,AgentOutput,get_selected_sequence,None,Thought: The current language of the user is: ...
6,2026-07-28T01:33:10.251369+00:00,203,SiteEvidenceAgent,ToolCall,get_selected_sequence,"{""n"": 5}",None
7,2026-07-28T01:33:10.261553+00:00,204,SiteEvidenceAgent,ToolCallResult,get_selected_sequence,"{""n"": 5}","{'status': 'ok', 'returned': 5, 'available': 2..."
8,2026-07-28T01:33:45.812569+00:00,460,SiteEvidenceAgent,AgentOutput,get_candidate,None,Thought: I now have the selected candidate sit...
9,2026-07-28T01:33:46.070253+00:00,461,SiteEvidenceAgent,ToolCall,get_candidate,"{""candidate_id"": ""green_p_710""}",None


## 12. Validate the final structured response

The code first uses LlamaIndex's Pydantic helper. It then falls back to
`structured_response` or a JSON object extracted from the response text.


In [16]:
def extract_json_object(text: str) -> dict[str, Any] | None:
    matches = re.findall(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        text,
        flags=re.DOTALL,
    )
    matches += re.findall(r"(\{.*\})", text, flags=re.DOTALL)

    for candidate in matches:
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(payload, dict):
            return payload
    return None


def validate_response(response: Any):
    errors = []

    if hasattr(response, "get_pydantic_model"):
        try:
            return (
                response.get_pydantic_model(FinalPlanningReport),
                errors,
            )
        except Exception as error:
            errors.append(f"get_pydantic_model failed: {error}")

    structured = getattr(response, "structured_response", None)
    if structured is not None:
        try:
            return (
                FinalPlanningReport.model_validate(structured),
                errors,
            )
        except ValidationError as error:
            errors.append(
                f"structured_response validation failed: {error}"
            )

    payload = extract_json_object(str(response))
    if payload is not None:
        try:
            return (
                FinalPlanningReport.model_validate(payload),
                errors,
            )
        except ValidationError as error:
            errors.append(f"JSON validation failed: {error}")
    else:
        errors.append("No JSON object was found in the response.")

    return None, errors


In [17]:
def build_deterministic_final_report(
    *,
    toolbox: GeoPlanToolbox,
    requested_count: int,
    requested_scenario: str,
) -> FinalPlanningReport:
    """
    Build the authoritative final report directly from GeoPlan tools.

    This is used when the multi-agent workflow ends on an intermediate
    handoff message instead of producing FinalPlanningReport.
    """

    sequence_result = toolbox.get_selected_sequence(
        n=requested_count
    )

    if sequence_result.get("status") != "ok":
        raise RuntimeError(
            "Could not retrieve the deterministic selected sequence."
        )

    selected_records = sequence_result["selection"]
    recommendations = []
    candidate_warnings = []

    for record in selected_records:
        candidate_id = record["candidate_id"]

        candidate_result = toolbox.get_candidate(
            candidate_id
        )

        if candidate_result.get("status") != "ok":
            raise RuntimeError(
                f"Could not retrieve candidate {candidate_id}."
            )

        candidate = candidate_result["candidate"]

        audit_result = toolbox.audit_candidate(
            candidate_id
        )

        critical_issues = (
            audit_result.get("critical_issues")
            or []
        )
        warnings = (
            audit_result.get("warnings")
            or []
        )

        concerns = list(
            dict.fromkeys(
                critical_issues + warnings
            )
        )

        candidate_warnings.extend(concerns)

        selection_score = record.get(
            "selection_score",
            candidate.get("selection_score"),
        )
        marginal_population = record.get(
            "marginal_population"
        )
        capacity_filled = record.get(
            "capacity_filled",
            candidate.get("capacity_filled"),
        )
        overall_score = record.get(
            "overall_score",
            candidate.get("overall_score"),
        )

        strengths = []

        if selection_score is not None:
            strengths.append(
                "selection_score="
                f"{float(selection_score):.10g}"
            )

        if marginal_population is not None:
            strengths.append(
                "marginal_population="
                f"{float(marginal_population):.10g}"
            )

        if capacity_filled is not None:
            strengths.append(
                "capacity_filled="
                f"{float(capacity_filled):.10g}"
            )

        if overall_score is not None:
            strengths.append(
                "overall_score="
                f"{float(overall_score):.10g}"
            )

        recommendations.append(
            CandidateRecommendation(
                candidate_id=candidate_id,
                address=record.get(
                    "address",
                    candidate.get("address"),
                ),
                selection_round=record.get(
                    "selection_round"
                ),
                selection_score=selection_score,
                marginal_population=(
                    marginal_population
                ),
                capacity_filled=capacity_filled,
                overall_score=overall_score,
                strengths=strengths,
                concerns=concerns,
                evidence_sources=[
                    "get_selected_sequence",
                    "get_candidate",
                    "audit_candidate",
                ],
            )
        )

    selected_ids = [
        site.candidate_id
        for site in recommendations
    ]

    scenario_result = toolbox.get_scenario_results(
        scenario_name=requested_scenario,
        n=requested_count,
    )

    sensitivity_result = (
        toolbox.get_sensitivity_results(
            n=requested_count
        )
    )

    recommendation_audit = (
        toolbox.audit_recommendation_set(
            candidate_ids=selected_ids
        )
    )

    scenario_available = (
        scenario_result.get("status") == "ok"
    )
    sensitivity_available = (
        sensitivity_result.get("status") == "ok"
    )

    robustness_findings = []

    if scenario_available:
        robustness_findings.append(
            "Scenario analysis was available from "
            "get_scenario_results."
        )
    else:
        robustness_findings.append(
            "Scenario analysis was not evaluated because "
            "the scenario export was unavailable."
        )

    if sensitivity_available:
        robustness_findings.append(
            "Sensitivity analysis was available from "
            "get_sensitivity_results."
        )
    else:
        robustness_findings.append(
            "Sensitivity analysis was not evaluated because "
            "the sensitivity export was unavailable."
        )

    audit_critical_issues = (
        recommendation_audit.get(
            "critical_issues"
        )
        or []
    )
    audit_warnings = (
        recommendation_audit.get("warnings")
        or []
    )

    data_quality_warnings = list(
        dict.fromkeys(
            candidate_warnings
            + audit_warnings
        )
    )

    reviewer_approved = (
        recommendation_audit.get(
            "reviewer_approved"
        )
        is True
    )

    specialist_findings = [
        SpecialistFinding(
            agent_name="SiteEvidenceAgent",
            findings=[
                (
                    f"Retrieved deterministic evidence for "
                    f"{len(recommendations)} selected candidates."
                )
            ],
            warnings=[],
            evidence_tools=[
                "get_selected_sequence",
                "get_candidate",
            ],
        ),
        SpecialistFinding(
            agent_name="ScenarioRiskAgent",
            findings=robustness_findings,
            warnings=audit_warnings,
            evidence_tools=[
                "get_scenario_results",
                "get_sensitivity_results",
                "audit_recommendation_set",
            ],
        ),
        SpecialistFinding(
            agent_name="FinalReviewerAgent",
            findings=[
                (
                    "Deterministic recommendation audit "
                    f"approved={reviewer_approved}."
                )
            ],
            warnings=audit_critical_issues,
            evidence_tools=[
                "audit_recommendation_set"
            ],
        ),
    ]

    limitations = [
        (
            "The recommendations are planning-screening "
            "results, not confirmed engineering feasibility."
        ),
        (
            "Electrical capacity and utility-connection "
            "availability were not evaluated."
        ),
        (
            "Construction cost, ownership, permitting, and "
            "detailed site design were not evaluated."
        ),
        (
            "Results depend on available datasets, scoring "
            "weights, and spatial thresholds."
        ),
    ]

    return FinalPlanningReport(
        objective="balanced planning objective",
        scenario=requested_scenario,
        selection_method=(
            "deterministic selected sequence"
        ),
        requested_site_count=requested_count,
        recommended_sites=recommendations,
        specialist_findings=specialist_findings,
        robustness_findings=robustness_findings,
        data_quality_warnings=data_quality_warnings,
        limitations=limitations,
        reviewer_approved=reviewer_approved,
    )


final_report, structure_errors = validate_response(
    workflow_response
)

if final_report is None:
    print(
        "The workflow ended without a structured final report."
    )

    for error in structure_errors:
        print(" -", error)

    print(
        "\nIntermediate workflow response:"
    )
    print(str(workflow_response))

    structure_errors.append(
        "Deterministic fallback report was used because "
        "the workflow ended on an intermediate response."
    )

    final_report = (
        build_deterministic_final_report(
            toolbox=toolbox,
            requested_count=DEMO_SITE_COUNT,
            requested_scenario=DEMO_SCENARIO,
        )
    )

    print(
        "\nBuilt FinalPlanningReport from deterministic "
        "GeoPlan tools."
    )
else:
    print(
        "The workflow returned a structured "
        "FinalPlanningReport."
    )

print(
    final_report.model_dump_json(
        indent=2
    )
)


The workflow ended without a structured final report.

Intermediate workflow response:
The request has been successfully processed and handed off to the PlanningCoordinator for final review and report generation. All scenario results, sensitivity analysis, and audit validations are complete and consistent. The balanced scenario demonstrates robustness with high-frequency selections and no critical issues. All candidate sites meet planning screening criteria with low risk levels and adequate spacing. The final report will be generated by the PlanningCoordinator based on this comprehensive analysis.

Built FinalPlanningReport from deterministic GeoPlan tools.
{
  "objective": "balanced planning objective",
  "scenario": "balanced",
  "selection_method": "deterministic selected sequence",
  "requested_site_count": 5,
  "recommended_sites": [
    {
      "candidate_id": "green_p_710",
      "address": "100 Grangeway Avenue",
      "selection_round": 1,
      "selection_score": 0.7307893907

In [18]:
import math
from typing import Any

import pandas as pd


def canonicalize_final_report(
    report: FinalPlanningReport,
    *,
    toolbox: GeoPlanToolbox,
    requested_count: int,
    requested_scenario: str,
) -> FinalPlanningReport:
    """
    Replace LLM-generated factual fields with exact deterministic values.

    The LLM may help with coordination and narrative, but Python remains
    authoritative for IDs, order, addresses, metrics, warnings, and approval.
    """

    sequence_result = toolbox.get_selected_sequence(
        n=requested_count
    )

    if sequence_result.get("status") != "ok":
        raise RuntimeError(
            "Could not retrieve the deterministic selected sequence."
        )

    selected_records = sequence_result["selection"]

    grounded_sites = []
    all_candidate_warnings = []

    for record in selected_records:
        candidate_id = record["candidate_id"]

        candidate_result = toolbox.get_candidate(
            candidate_id
        )

        if candidate_result.get("status") != "ok":
            raise RuntimeError(
                f"Could not retrieve candidate {candidate_id}."
            )

        candidate = candidate_result["candidate"]

        candidate_audit = toolbox.audit_candidate(
            candidate_id
        )

        concerns = list(
            dict.fromkeys(
                candidate_audit.get(
                    "critical_issues",
                    [],
                )
                + candidate_audit.get(
                    "warnings",
                    [],
                )
            )
        )

        all_candidate_warnings.extend(concerns)

        selection_score = record.get(
            "selection_score"
        )
        marginal_population = record.get(
            "marginal_population"
        )
        capacity_filled = record.get(
            "capacity_filled",
            candidate.get("capacity_filled"),
        )
        overall_score = record.get(
            "overall_score",
            candidate.get("overall_score"),
        )

        # Deterministically generated, fully grounded strengths.
        strengths = []

        if selection_score is not None:
            strengths.append(
                "Selection score: "
                f"{float(selection_score):.6f}"
            )

        if marginal_population is not None:
            strengths.append(
                "Marginal population covered: "
                f"{float(marginal_population):,.0f}"
            )

        if capacity_filled is not None:
            strengths.append(
                "Parking capacity used by the model: "
                f"{float(capacity_filled):,.0f}"
            )

        if overall_score is not None:
            strengths.append(
                "Overall suitability score: "
                f"{float(overall_score):.6f}"
            )

        grounded_sites.append(
            CandidateRecommendation(
                candidate_id=candidate_id,
                address=record.get(
                    "address",
                    candidate.get("address"),
                ),
                selection_round=record.get(
                    "selection_round"
                ),
                selection_score=selection_score,
                marginal_population=marginal_population,
                capacity_filled=capacity_filled,
                overall_score=overall_score,
                strengths=strengths,
                concerns=concerns,
                evidence_sources=[
                    "get_selected_sequence",
                    "get_candidate",
                    "audit_candidate",
                ],
            )
        )

    recommended_ids = [
        site.candidate_id
        for site in grounded_sites
    ]

    deterministic_audit = (
        toolbox.audit_recommendation_set(
            candidate_ids=recommended_ids
        )
    )

    scenario_result = toolbox.get_scenario_results(
        scenario_name=requested_scenario,
        n=requested_count,
    )

    sensitivity_result = (
        toolbox.get_sensitivity_results(
            n=requested_count
        )
    )

    if (
        scenario_result.get("status") != "ok"
        and sensitivity_result.get("status") != "ok"
    ):
        robustness_findings = [
            (
                "Scenario and sensitivity robustness were not "
                "evaluated because the optional exports were unavailable."
            )
        ]
    else:
        robustness_findings = (
            report.robustness_findings
        )

    # Remove the incorrect statement that marginal_population
    # is unavailable.
    cleaned_warnings = [
        warning
        for warning in report.data_quality_warnings
        if not (
            "marginal_population" in warning.lower()
            and "not available" in warning.lower()
        )
    ]

    cleaned_warnings.extend(
        all_candidate_warnings
    )

    cleaned_warnings = list(
        dict.fromkeys(cleaned_warnings)
    )

    required_limitations = [
        (
            "The recommendations represent planning screening, "
            "not confirmed engineering feasibility."
        ),
        (
            "Electrical capacity and utility-connection availability "
            "were not evaluated."
        ),
        (
            "Construction cost, ownership, permitting, and detailed "
            "site implementation constraints were not evaluated."
        ),
        (
            "Results depend on the available datasets, model assumptions, "
            "scoring weights, and spatial thresholds."
        ),
    ]

    limitations = list(
        dict.fromkeys(
            report.limitations
            + required_limitations
        )
    )

    return report.model_copy(
        update={
            "scenario": requested_scenario,
            "selection_method": (
                "deterministic selected sequence"
            ),
            "requested_site_count": (
                requested_count
            ),
            "recommended_sites": grounded_sites,
            "robustness_findings": (
                robustness_findings
            ),
            "data_quality_warnings": (
                cleaned_warnings
            ),
            "limitations": limitations,
            "reviewer_approved": (
                deterministic_audit.get(
                    "reviewer_approved"
                )
                is True
            ),
        }
    )


def values_match(
    reported: float | int | None,
    expected: float | int | None,
    tolerance: float = 1e-9,
) -> bool:
    if reported is None and expected is None:
        return True

    if reported is None or expected is None:
        return False

    try:
        return math.isclose(
            float(reported),
            float(expected),
            rel_tol=tolerance,
            abs_tol=tolerance,
        )
    except (TypeError, ValueError):
        return False


def validate_report_grounding(
    report: FinalPlanningReport,
    *,
    requested_count: int,
    requested_scenario: str,
) -> dict[str, Any]:
    errors = []
    warnings = []

    expected_result = toolbox.get_selected_sequence(
        n=requested_count
    )

    if expected_result.get("status") != "ok":
        return {
            "grounded": False,
            "errors": [
                "Could not retrieve the deterministic selected sequence."
            ],
            "warnings": [],
        }

    expected_records = expected_result["selection"]

    expected_ids = [
        record["candidate_id"]
        for record in expected_records
    ]

    reported_ids = [
        site.candidate_id
        for site in report.recommended_sites
    ]

    if (
        report.scenario.strip().lower()
        != requested_scenario.strip().lower()
    ):
        errors.append(
            f"Scenario must be {requested_scenario!r}."
        )

    if (
        report.selection_method.strip().lower()
        != "deterministic selected sequence"
    ):
        errors.append(
            "selection_method is incorrect."
        )

    if report.requested_site_count != requested_count:
        errors.append(
            "requested_site_count is incorrect."
        )

    if len(report.recommended_sites) != requested_count:
        errors.append(
            "The returned site count is incorrect."
        )

    if reported_ids != expected_ids:
        errors.append(
            "Candidate IDs or their order do not match "
            "get_selected_sequence."
        )

    if len(reported_ids) != len(set(reported_ids)):
        errors.append(
            "The report contains duplicate IDs."
        )

    expected_by_id = {
        record["candidate_id"]: record
        for record in expected_records
    }

    for site in report.recommended_sites:
        expected = expected_by_id.get(
            site.candidate_id
        )

        if expected is None:
            errors.append(
                f"{site.candidate_id}: unexpected candidate ID."
            )
            continue

        candidate_result = toolbox.get_candidate(
            site.candidate_id
        )

        candidate = candidate_result.get(
            "candidate",
            {},
        )

        expected_address = expected.get(
            "address",
            candidate.get("address"),
        )

        if site.address != expected_address:
            errors.append(
                f"{site.candidate_id}: address mismatch."
            )

        if (
            site.selection_round
            != expected.get("selection_round")
        ):
            errors.append(
                f"{site.candidate_id}: selection-round mismatch."
            )

        numeric_checks = {
            "selection_score": expected.get(
                "selection_score"
            ),
            "marginal_population": expected.get(
                "marginal_population"
            ),
            "capacity_filled": expected.get(
                "capacity_filled",
                candidate.get("capacity_filled"),
            ),
            "overall_score": expected.get(
                "overall_score",
                candidate.get("overall_score"),
            ),
        }

        for field_name, expected_value in (
            numeric_checks.items()
        ):
            reported_value = getattr(
                site,
                field_name,
            )

            if not values_match(
                reported_value,
                expected_value,
            ):
                errors.append(
                    f"{site.candidate_id}: "
                    f"{field_name} mismatch."
                )

        if not site.evidence_sources:
            errors.append(
                f"{site.candidate_id}: "
                "evidence_sources is empty."
            )

    if len(report.limitations) < 3:
        errors.append(
            "At least three limitations are required."
        )

    incorrect_marginal_warning = any(
        "marginal_population" in warning.lower()
        and "not available" in warning.lower()
        for warning in report.data_quality_warnings
    )

    if incorrect_marginal_warning:
        errors.append(
            "The report incorrectly says marginal_population "
            "is unavailable."
        )

    deterministic_audit = (
        toolbox.audit_recommendation_set(
            candidate_ids=reported_ids
        )
    )

    audit_approved = (
        deterministic_audit.get(
            "reviewer_approved"
        )
        is True
    )

    if report.reviewer_approved != audit_approved:
        errors.append(
            "reviewer_approved does not match "
            "the deterministic audit."
        )

    return {
        "grounded": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
        "expected_ids": expected_ids,
        "reported_ids": reported_ids,
        "deterministic_audit": deterministic_audit,
    }


# ------------------------------------------------------------
# Replace LLM-generated factual fields with authoritative values.
# ------------------------------------------------------------

final_report = canonicalize_final_report(
    final_report,
    toolbox=toolbox,
    requested_count=DEMO_SITE_COUNT,
    requested_scenario=DEMO_SCENARIO,
)

print("CANONICAL FINAL REPORT")
print("=" * 70)

print(
    final_report.model_dump_json(
        indent=2
    )
)


# ------------------------------------------------------------
# Validate the corrected report.
# ------------------------------------------------------------

grounding_validation = validate_report_grounding(
    final_report,
    requested_count=DEMO_SITE_COUNT,
    requested_scenario=DEMO_SCENARIO,
)

checks = {
    "pydantic_valid": True,
    "grounding_valid": (
        grounding_validation["grounded"]
    ),
    "scenario_matches_request": (
        final_report.scenario.strip().lower()
        == DEMO_SCENARIO.strip().lower()
    ),
    "selection_method_correct": (
        final_report.selection_method.strip().lower()
        == "deterministic selected sequence"
    ),
    "site_count_matches_request": (
        len(final_report.recommended_sites)
        == DEMO_SITE_COUNT
    ),
    "candidate_ids_match_sequence": (
        grounding_validation["reported_ids"]
        == grounding_validation["expected_ids"]
    ),
    "limitations_present": (
        len(final_report.limitations) >= 3
    ),
    "reviewer_approved": (
        final_report.reviewer_approved
    ),
}

display(
    pd.DataFrame(
        [
            {
                "check": key,
                "value": value,
            }
            for key, value in checks.items()
        ]
    )
)

if grounding_validation["warnings"]:
    print("\nGrounding warnings:")

    for warning in grounding_validation["warnings"]:
        print(" -", warning)

if grounding_validation["errors"]:
    print("\nGrounding errors:")

    for error in grounding_validation["errors"]:
        print(" -", error)

if not grounding_validation["grounded"]:
    raise AssertionError(
        "The canonical final report still failed "
        "deterministic grounding validation."
    )

print(
    "\nAll structured and evidence-grounding checks passed."
)

CANONICAL FINAL REPORT
{
  "objective": "balanced planning objective",
  "scenario": "balanced",
  "selection_method": "deterministic selected sequence",
  "requested_site_count": 5,
  "recommended_sites": [
    {
      "candidate_id": "green_p_710",
      "address": "100 Grangeway Avenue",
      "selection_round": 1,
      "selection_score": 0.7307893907056662,
      "marginal_population": 17892.0,
      "capacity_filled": 214.0,
      "overall_score": 0.7877772435089739,
      "strengths": [
        "Selection score: 0.730789",
        "Marginal population covered: 17,892",
        "Parking capacity used by the model: 214",
        "Overall suitability score: 0.787777"
      ],
      "concerns": [],
      "evidence_sources": [
        "get_selected_sequence",
        "get_candidate",
        "audit_candidate"
      ]
    },
    {
      "candidate_id": "green_p_821",
      "address": "Kennedy South Lot -155 Transway Cres",
      "selection_round": 2,
      "selection_score": 0.7839313

,check,value
0,pydantic_valid,True
1,grounding_valid,True
2,scenario_matches_request,True
3,selection_method_correct,True
4,site_count_matches_request,True
5,candidate_ids_match_sequence,True
6,limitations_present,True
7,reviewer_approved,True



All structured and evidence-grounding checks passed.


## 13. Save the final report and validation results


In [19]:
FINAL_REPORT_PATH = (
    OUTPUT_DIR / "final_agent_report.json"
)
VALIDATION_PATH = (
    OUTPUT_DIR
    / "final_agent_report_validation.json"
)
GROUNDING_PATH = (
    OUTPUT_DIR
    / "final_agent_report_grounding.json"
)

with FINAL_REPORT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_report.model_dump(mode="json"),
        file,
        indent=2,
    )

with GROUNDING_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        grounding_validation,
        file,
        indent=2,
        default=str,
    )

with VALIDATION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "validated_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "model_name": MODEL_NAME,
            "requested_scenario": DEMO_SCENARIO,
            "requested_site_count": DEMO_SITE_COUNT,
            "checks": checks,
            "structure_errors_encountered": (
                structure_errors
            ),
            "grounding_valid": (
                grounding_validation["grounded"]
            ),
        },
        file,
        indent=2,
        default=str,
    )

print("Saved:", FINAL_REPORT_PATH.resolve())
print("Saved:", VALIDATION_PATH.resolve())
print("Saved:", GROUNDING_PATH.resolve())
print("Saved:", TRACE_PATH.resolve())


Saved: /Users/miladsaeedi/Desktop/Daily_Work_load/Projects_for_CV/Geospatial_site_selection/outputs/final_agent_report.json
Saved: /Users/miladsaeedi/Desktop/Daily_Work_load/Projects_for_CV/Geospatial_site_selection/outputs/final_agent_report_validation.json
Saved: /Users/miladsaeedi/Desktop/Daily_Work_load/Projects_for_CV/Geospatial_site_selection/outputs/final_agent_report_grounding.json
Saved: /Users/miladsaeedi/Desktop/Daily_Work_load/Projects_for_CV/Geospatial_site_selection/outputs/multiagent_trace.csv


# Workflow complete

A successful run now demonstrates two different kinds of validation:

1. **Pydantic validation**  
   The final output has the required structure and data types.

2. **Deterministic grounding validation**  
   Candidate IDs, sequence, addresses, rounds, numeric evidence,
   scenario, reviewer decision, and limitations match the GeoPlan tools.

## Expected output files

```text
outputs/
├── multiagent_trace.csv
├── final_agent_report.json
├── final_agent_report_validation.json
└── final_agent_report_grounding.json
```

The final report is not accepted merely because it looks reasonable.
It must agree with the deterministic evidence layer.
